# Entraînement final et exploration du modèle BERTopic

Ce notebook recharge les paramètres retenus lors de l'optimisation, entraîne le modèle final et explore la structure thématique du corpus.

In [20]:
import gc
import time
import warnings
from itertools import combinations
from pathlib import Path #pour la gestion des chemins de fichiers
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import ParameterGrid
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.feature_extraction.text import CountVectorizer

from umap import UMAP
from hdbscan import HDBSCAN
from hdbscan.validity import validity_index

from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sentence_transformers import SentenceTransformer #pour les embeddings de phrase

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

COL_TEXTE = "phrases_lemm"
COL_ROMAN = "roman"

warnings.filterwarnings("ignore")

## 1. Chargement du corpus et chargement ou calcul des embeddings

In [21]:
# Racine du projet ZOLA-DTM
PROJECT_ROOT = Path.cwd().parent

# Dossiers principaux
DATA_DIR = PROJECT_ROOT / "data"
DONNEES_ANNEX_DIR = DATA_DIR / "donnees_annex"
CACHE_DIR = DONNEES_ANNEX_DIR / "data_cache"

# Fichier des embeddings
CHEMIN_EMBEDDINGS = CACHE_DIR / "embeddings_sentence_camembert.npy"

print("Répertoire courant :", Path.cwd())
print("Racine projet       :", PROJECT_ROOT)
print("Dossier data        :", DATA_DIR)
print("Dossier cache       :", CACHE_DIR)
print("Embeddings          :", CHEMIN_EMBEDDINGS)

print("==========================================")
print("Vérification de l'existence des fichiers :")
chemin_csv = Path("../data/2_processed/03_corpus_lematise_128.csv")
print("cwd =", Path.cwd())
print("chemin_csv =", chemin_csv)
print("resolve =", chemin_csv.resolve())
print("exists =", chemin_csv.exists())

Répertoire courant : /Users/morganrichard/zola-dtm/02_notebook_analyse
Racine projet       : /Users/morganrichard/zola-dtm
Dossier data        : /Users/morganrichard/zola-dtm/data
Dossier cache       : /Users/morganrichard/zola-dtm/data/donnees_annex/data_cache
Embeddings          : /Users/morganrichard/zola-dtm/data/donnees_annex/data_cache/embeddings_sentence_camembert.npy
Vérification de l'existence des fichiers :
cwd = /Users/morganrichard/zola-dtm/02_notebook_analyse
chemin_csv = ../data/2_processed/03_corpus_lematise_128.csv
resolve = /Users/morganrichard/zola-dtm/data/2_processed/03_corpus_lematise_128.csv
exists = True


In [22]:
df=pd.read_csv(Path(chemin_csv, encoding="utf-8"))

CHEMIN_EMBEDDINGS = Path("../data/donnees_annex/data_cache/embeddings_sentence_camembert.npy")

if CHEMIN_EMBEDDINGS.exists():
    print("Chargement des embeddings sauvegardés...")
    embeddings = np.load(CHEMIN_EMBEDDINGS, allow_pickle=False)
else:
    embedding_model = SentenceTransformer("dangvantuan/sentence-camembert-base")

    print("Génération des embeddings sémantiques...")
    embeddings = embedding_model.encode(
        df["texte"].tolist(),
        batch_size=64,
        show_progress_bar=True
    )
    CHEMIN_EMBEDDINGS.parent.mkdir(parents=True, exist_ok=True)
    np.save(CHEMIN_EMBEDDINGS, embeddings)
    print(f"Embeddings sauvegardés dans : {CHEMIN_EMBEDDINGS}")

Chargement des embeddings sauvegardés...


## 2. Préparation des documents pour BERTopic

In [23]:
# Vérification des colonnes
assert COL_TEXTE in df.columns, (
    f"La colonne '{COL_TEXTE}' n'existe pas dans df."
)
assert COL_ROMAN in df.columns, (
    f"La colonne '{COL_ROMAN}' n'existe pas dans df."
)

# Conversion des embeddings
embeddings_array_initial = np.asarray(embeddings)

assert len(df) == len(embeddings_array_initial), (
    "Le nombre de lignes de df ne correspond pas "
    "au nombre d'embeddings."
)

# Masque des documents non vides
masque_documents = (df[COL_TEXTE].notna() & df[COL_TEXTE].astype(str).str.strip().ne(""))

# DataFrame utilisé par BERTopic
df_model = (df.loc[masque_documents].copy().reset_index(drop=True))

# Documents
documents = (df_model[COL_TEXTE].astype(str).tolist())

# Romans associés aux documents
romans = (df_model[COL_ROMAN].astype(str).tolist())

# Embeddings alignés
embeddings_array = embeddings_array_initial[masque_documents.to_numpy()]

# Tokenisation simple pour la cohérence C_v
texts_tokenises = [
    document.split()
    for document in documents
]

# Dictionnaire Gensim
dictionary = Dictionary(texts_tokenises)


print("Nombre de documents :", len(documents))
print("Nombre de romans :", df_model[COL_ROMAN].nunique())
print("Dimensions des embeddings :", embeddings_array.shape)
print("Taille du dictionnaire :", len(dictionary))

Nombre de documents : 29421
Nombre de romans : 31
Dimensions des embeddings : (29421, 768)
Taille du dictionnaire : 19176


## 3. Fonctions d'évaluation finale

In [35]:
def extraire_mots_topics(
    topic_model,
    labels,
    top_n_words=15,
    dictionary=None
):
    """
    Extrait les mots des topics, en excluant le topic -1.

    Si un dictionnaire Gensim est fourni, seuls les mots présents
    dans ce dictionnaire sont conservés.
    """

    labels = np.asarray(labels)

    topic_ids = sorted(
        int(topic_id)
        for topic_id in np.unique(labels)
        if topic_id != -1
    )

    topics_words = []

    for topic_id in topic_ids:

        representation = topic_model.get_topic(topic_id)

        if not representation:
            continue

        words = [
            word
            for word, _ in representation[:top_n_words]
        ]

        if dictionary is not None:
            words = [
                word
                for word in words
                if word in dictionary.token2id
            ]

        # Éviter les topics insuffisamment représentés
        if len(words) >= 2:
            topics_words.append(words)

    return topics_words


def calculer_coherence_cv(
    topic_model,
    labels,
    texts_tokenises,
    dictionary,
    top_n_words=10
):
    """
    Calcule la cohérence C_v des topics BERTopic.
    """

    topics_words = extraire_mots_topics(
        topic_model=topic_model,
        labels=labels,
        top_n_words=top_n_words,
        dictionary=dictionary
    )

    if len(topics_words) < 2:
        return np.nan

    try:
        coherence_model = CoherenceModel(
            topics=topics_words,
            texts=texts_tokenises,
            dictionary=dictionary,
            coherence="c_v",
            topn=top_n_words,
            processes=1
        )

        return float(coherence_model.get_coherence())

    except Exception:
        return np.nan

def calculer_diversite_topics(
    topic_model,
    labels,
    top_n_words=10
):
    """
    Diversité lexicale :
    nombre de termes uniques / nombre total de termes.
    """

    topics_words = extraire_mots_topics(
        topic_model=topic_model,
        labels=labels,
        top_n_words=top_n_words,
        dictionary=None
    )

    tous_les_mots = [
        word
        for topic_words in topics_words
        for word in topic_words
    ]

    if not tous_les_mots:
        return np.nan

    return len(set(tous_les_mots)) / len(tous_les_mots)
    

## 4. Chargement des paramètres et entraînement du modèle final

Les paramètres sélectionnés dans le notebook d'optimisation sont rechargés depuis le fichier JSON.

In [36]:
CHEMIN_PARAMETRES = Path("../data/donnees_annex/meilleurs_parametres_bertopic.json")

with CHEMIN_PARAMETRES.open("r", encoding="utf-8") as f:
    best_params = json.load(f)

best_umap_model = UMAP(
    n_neighbors=best_params["n_neighbors"],
    n_components=best_params["n_components"],
    min_dist=0.0,
    metric="cosine",
    random_state=best_params["random_state"],
    low_memory=True
)

best_hdbscan_model = HDBSCAN(
    min_cluster_size=best_params["min_cluster_size"],
    min_samples=best_params["min_samples"],
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
    core_dist_n_jobs=-1
)

best_vectorizer_model = CountVectorizer(
    min_df=2,
    max_df=0.8,
)

best_ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True,
    bm25_weighting=True
)

best_topic_model = BERTopic(
    language="french",
    hdbscan_model=best_hdbscan_model,
    umap_model=best_umap_model,
    vectorizer_model=best_vectorizer_model,
    ctfidf_model=best_ctfidf_model,
    calculate_probabilities=False,
    nr_topics=None,
    verbose=True
)

best_topics_raw, _ = best_topic_model.fit_transform(
    documents,
    embeddings=embeddings_array
)

best_topics_raw = np.asarray(best_topics_raw)


nombre_topics_bruts = len(
    np.unique(
        best_topics_raw[
            best_topics_raw != -1
        ]
    )
)

taux_outliers_brut = np.mean(best_topics_raw == -1)

print("Nombre naturel de topics :", nombre_topics_bruts)

print( f"Taux brut d'outliers : " f"{taux_outliers_brut:.1%}")

display(best_topic_model.get_topic_info())

2026-08-12 14:19:03,842 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-12 14:19:35,962 - BERTopic - Dimensionality - Completed ✓
2026-08-12 14:19:35,963 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-12 14:19:37,452 - BERTopic - Cluster - Completed ✓
2026-08-12 14:19:37,455 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-12 14:19:37,695 - BERTopic - Representation - Completed ✓


Nombre naturel de topics : 18
Taux brut d'outliers : 64.6%


,Topic,Count,Name,Representation,Representative_Docs
0,-1,18998,-1_ménage_dîner_hôtel_gentil,"[ménage, dîner, hôtel, gentil, après, souffle,...",[grâce divin guérie amour reconnaissance rampe...
1,0,3245,0_horizon_herbe_verdure_armée,"[horizon, herbe, verdure, armée, colonne, géan...",[massif allée tour serre gradin demi tuyau cha...
2,1,1303,1_chéri_écoute_baiser_tai,"[chéri, écoute, baiser, tai, cruel, veu, étrei...",[gorge émotion poitrine meurtrir caresse fatal...
3,2,1272,2_peuple_nation_siècle_pape,"[peuple, nation, siècle, pape, catholicisme, s...",[ignorant inquiet génie oracle antique éternit...
4,3,1128,3_baiser_étreinte_volupté_épouvante,"[baiser, étreinte, volupté, épouvante, affecti...",[épouvante amant cauchemar baiser insomnie pru...
5,4,501,4_cardinal_huissier_ministre_eminence,"[cardinal, huissier, ministre, eminence, évêqu...",[tranquille obligeance phrase prêtre secrétair...
6,5,377,5_rente_aîné_âgé_bénéfice,"[rente, aîné, âgé, bénéfice, cadet, million, h...",[mois mort catastrophe conseil villa million c...
7,6,365,6_sommeil_oreiller_bougie_couverture,"[sommeil, oreiller, bougie, couverture, insomn...",[instant soirée mot bourgeois idée extraordina...
8,7,281,7_demoiselle_économie_comptoir_vendeur,"[demoiselle, économie, comptoir, vendeur, ména...",[reine fantasque époque domination cabinet toi...
9,8,280,8_rein_balle_cuisse_coude,"[rein, balle, cuisse, coude, bond, sueur, seco...",[terrible bûcheron cognée fort charpente fer f...


## 5. Réaffectation des documents hors topic

Plusieurs seuils sont comparés avant de retenir le seuil final de réaffectation des outliers.

In [37]:
seuils_outliers = [
    0.0,
    0.10,
    0.20,
    0.30,
    0.40,
    0.50
]

comparaisons_seuils = []
topics_par_seuil = {}

for seuil in seuils_outliers:
    topics_test = best_topic_model.reduce_outliers(
        documents,
        best_topics_raw,
        strategy="embeddings",
        embeddings=embeddings_array,
        threshold=seuil)

    topics_test = np.asarray(topics_test)

    topics_par_seuil[seuil] = topics_test

    masque_assignes = topics_test != -1

    tailles_topics = (pd.Series(topics_test[masque_assignes]).value_counts())

    nombre_assignes = int(masque_assignes.sum())

    nombre_reassignes = int(((best_topics_raw == -1) & (topics_test != -1)).sum())

    comparaisons_seuils.append({
        "threshold": seuil,
        "n_topics": len(tailles_topics),
        "outlier_rate_remaining": np.mean(
            topics_test == -1
        ),
        "n_reassigned": nombre_reassignes,
        "smallest_topic": (
            tailles_topics.min()
            if len(tailles_topics) > 0
            else np.nan
        ),
        "largest_topic": (
            tailles_topics.max()
            if len(tailles_topics) > 0
            else np.nan
        ),
        "largest_topic_share": (
            tailles_topics.max() / nombre_assignes
            if nombre_assignes > 0
            else np.nan
        )
    })


comparaison_seuils_df = pd.DataFrame(comparaisons_seuils)

display(
    comparaison_seuils_df.style.format({
        "threshold": "{:.2f}",
        "outlier_rate_remaining": "{:.1%}",
        "largest_topic_share": "{:.1%}"
    })
)

,threshold,n_topics,outlier_rate_remaining,n_reassigned,smallest_topic,largest_topic,largest_topic_share
0,0.00,18,0.0%,18998,677,4582,15.6%
1,0.10,18,0.0%,18998,677,4582,15.6%
2,0.20,18,0.0%,18998,677,4582,15.6%
3,0.30,18,0.0%,18998,677,4582,15.6%
4,0.40,18,0.0%,18998,677,4582,15.6%
5,0.50,18,0.1%,18980,676,4581,15.6%


In [38]:
SEUIL_OUTLIERS_FINAL = 0.30

topics_finaux = np.asarray(topics_par_seuil[SEUIL_OUTLIERS_FINAL])

print(
    f"Taux d'outliers final : "
    f"{np.mean(topics_finaux == -1):.1%}"
)


print(
    "Nombre final de topics :",
    len(np.unique(topics_finaux[topics_finaux != -1])))

distribution_topics = (
    pd.Series(topics_finaux)
    .value_counts()
    .sort_index()
    .rename_axis("Topic")
    .reset_index(name="Count")
)

display(distribution_topics)

Taux d'outliers final : 0.0%
Nombre final de topics : 18


,Topic,Count
0,0,4582
1,1,1927
2,2,2109
3,3,2608
4,4,1656
5,5,1624
6,6,1495
7,7,2055
8,8,1373
9,9,1261


## 6. Raffinement de la représentation lexicale des topics

In [48]:
stopwords_corpus = ["deberl", "men", "yole","embrass", "revien", "tai", "connai", "quidquid", "trouche", "hattoy", "sai", 'fasse','interrompit',
                    'rauque', 'pan', 'croyez', 'faite' ,'rêv', 'moi', 
                    ]

vectorizer_final = CountVectorizer(
    stop_words=stopwords_corpus,
    min_df=2,
    max_df=0.80,
    #ngram_range=(1, 2)
)

ctfidf_final = ClassTfidfTransformer(
    reduce_frequent_words=True,
    bm25_weighting=True
)

best_topic_model.update_topics(
    documents,
    topics=topics_finaux,
    vectorizer_model=vectorizer_final,
    ctfidf_model=ctfidf_final,
    top_n_words=15
)

display(best_topic_model.get_topic_info())

2026-08-12 14:21:11,614 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


,Topic,Count,Name,Representation,Representative_Docs
0,0,4582,0_toiture_rive_barricade_obu,"[toiture, rive, barricade, obu, vallée, feuill...",[massif allée tour serre gradin demi tuyau cha...
1,1,1927,1_pense_pleure_supplie_infamie,"[pense, pleure, supplie, infamie, aie, virgini...",[gorge émotion poitrine meurtrir caresse fatal...
2,2,2109,2_catholicisme_dogme_démocratie_papauté,"[catholicisme, dogme, démocratie, papauté, chr...",[ignorant inquiet génie oracle antique éternit...
3,3,2608,3_implacable_union_châtiment_arrachement,"[implacable, union, châtiment, arrachement, ch...",[épouvante amant cauchemar baiser insomnie pru...
4,4,1656,4_eminence_vicaire_obligeance_congrégation,"[eminence, vicaire, obligeance, congrégation, ...",[tranquille obligeance phrase prêtre secrétair...
5,5,1624,5_hectare_bail_enchère_entrepreneur,"[hectare, bail, enchère, entrepreneur, associé...",[mois mort catastrophe conseil villa million c...
6,6,1495,6_respiration_coucou_veilleuse_ronflement,"[respiration, coucou, veilleuse, ronflement, s...",[instant soirée mot bourgeois idée extraordina...
7,7,2055,7_blanchisseur_chapelier_zingueur_patronne,"[blanchisseur, chapelier, zingueur, patronne, ...",[reine fantasque époque domination cabinet toi...
8,8,1373,8_hurlement_fesse_hémorragie_raidie,"[hurlement, fesse, hémorragie, raidie, macquar...",[terrible bûcheron cognée fort charpente fer f...
9,9,1261,9_plaisant_drogue_clique_charcutier,"[plaisant, drogue, clique, charcutier, complim...",[taquin inquiet despotique lésion âge fou méch...


In [49]:
chemin_sortie = Path("..") /"data" /"4_resultats" /"informations_topics.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)
best_topic_model.get_topic_info().to_csv(chemin_sortie, index=False, encoding="utf-8")


## 7. Évaluation du modèle final

In [50]:
# L'analyseur produit exactement les tokens 
# attendus par le nouveau CountVectorizer.

analyseur_final = (vectorizer_final.build_analyzer())

texts_tokenises_final = [
    analyseur_final(document)
    for document in documents
]


dictionary_final = Dictionary(texts_tokenises_final)


coherence_finale = calculer_coherence_cv(
    topic_model=best_topic_model,
    labels=topics_finaux,
    texts_tokenises=texts_tokenises_final,
    dictionary=dictionary_final,
    top_n_words=10
)


diversite_finale = calculer_diversite_topics(
    topic_model=best_topic_model,
    labels=topics_finaux,
    top_n_words=10
)


print(
    f"Cohérence C_v finale : "
    f"{coherence_finale:.3f}"
)

print(
    f"Diversité finale : "
    f"{diversite_finale:.3f}"
)

Cohérence C_v finale : 0.435
Diversité finale : 0.961


## 8. Visualisations thématiques et temporelles

In [51]:
fig = best_topic_model.visualize_hierarchy()
fig.show()

In [52]:
# Calcul de l'évolution temporelle
topics_over_time = best_topic_model.topics_over_time(
    documents,
    df_model["annee"].tolist(),
    nr_bins=15
)

# Noms plus lisibles

# Création du graphique
fig = best_topic_model.visualize_topics_over_time(
    topics_over_time,
    custom_labels=True,
    normalize_frequency=False,
    title="Travail et milieux populaires",
    width=1500,
    height=700
)

# Personnalisation en français
fig.update_traces(mode="lines+markers")

fig.update_layout(
    template="plotly_white",
    xaxis_title="Période de publication",
    yaxis_title="Nombre de segments",
    legend_title_text=None
)



15it [00:00, 20.74it/s]


## 9. Analyse de la distribution des topics par roman

In [43]:
df_resultats = df_model.copy()

df_resultats["topic"] = topics_finaux

df_resultats["est_outlier"] = (df_resultats["topic"] == -1)

display(df_resultats[[COL_ROMAN, COL_TEXTE, "topic", "est_outlier"]].head())

,roman,phrases_lemm,topic,est_outlier
0,1865 La confession de Claude.,hiver matin frais manteau brouillard saison so...,0,False
1,1865 La confession de Claude.,soir vent porte mur flamme lampe ennui morne g...,0,False
2,1865 La confession de Claude.,chambre bel toile blanc meuble simple luisant ...,1,False
3,1865 La confession de Claude.,lèvre cœur reine laurier songe daignion règle ...,1,False
4,1865 La confession de Claude.,-vou brun rieur moisson vendange épi grappe se...,0,False


In [44]:
table_topics_romans_counts = pd.crosstab(df_resultats[COL_ROMAN],df_resultats["topic"])

display(table_topics_romans_counts)

topic,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
roman,,,,,,,,,,,,,,,,,,
1865 La confession de Claude.,46,181,3,19,2,1,24,6,4,9,5,0,2,0,2,0,30,1
1866 Le voeu d une morte.,19,35,3,84,11,23,12,11,7,15,7,22,0,2,10,0,19,1
1867 Les mysteres de Marseille.,109,54,33,76,130,101,31,22,46,28,21,30,86,32,40,36,22,7
1867 Therese Raquin.,29,37,4,144,4,21,70,21,40,11,23,21,5,1,14,3,17,1
1868 Madeleine Ferat.,52,107,6,233,15,30,60,23,35,10,19,19,2,1,26,3,44,3
Au Bonheur des dames.,169,36,10,59,28,40,23,139,20,38,166,109,38,82,53,6,23,1
Fecondite.,123,138,161,164,45,147,53,104,50,101,92,72,48,25,101,29,78,7
Germinal.,359,34,81,45,37,56,70,84,93,28,87,24,36,47,44,30,13,2
L argent.,84,27,85,59,77,86,12,52,14,41,44,50,79,199,28,15,24,7


In [45]:
table_topics_romans_proportions = pd.crosstab(
    df_resultats[COL_ROMAN],
    df_resultats["topic"],
    normalize="index"
)

display(table_topics_romans_proportions.style.format("{:.1%}"))

topic,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
roman,,,,,,,,,,,,,,,,,,
1865 La confession de Claude.,13.7%,54.0%,0.9%,5.7%,0.6%,0.3%,7.2%,1.8%,1.2%,2.7%,1.5%,0.0%,0.6%,0.0%,0.6%,0.0%,9.0%,0.3%
1866 Le voeu d une morte.,6.8%,12.5%,1.1%,29.9%,3.9%,8.2%,4.3%,3.9%,2.5%,5.3%,2.5%,7.8%,0.0%,0.7%,3.6%,0.0%,6.8%,0.4%
1867 Les mysteres de Marseille.,12.1%,6.0%,3.7%,8.4%,14.4%,11.2%,3.4%,2.4%,5.1%,3.1%,2.3%,3.3%,9.5%,3.5%,4.4%,4.0%,2.4%,0.8%
1867 Therese Raquin.,6.2%,7.9%,0.9%,30.9%,0.9%,4.5%,15.0%,4.5%,8.6%,2.4%,4.9%,4.5%,1.1%,0.2%,3.0%,0.6%,3.6%,0.2%
1868 Madeleine Ferat.,7.6%,15.6%,0.9%,33.9%,2.2%,4.4%,8.7%,3.3%,5.1%,1.5%,2.8%,2.8%,0.3%,0.1%,3.8%,0.4%,6.4%,0.4%
Au Bonheur des dames.,16.2%,3.5%,1.0%,5.7%,2.7%,3.8%,2.2%,13.4%,1.9%,3.7%,16.0%,10.5%,3.7%,7.9%,5.1%,0.6%,2.2%,0.1%
Fecondite.,8.0%,9.0%,10.5%,10.7%,2.9%,9.6%,3.4%,6.8%,3.3%,6.6%,6.0%,4.7%,3.1%,1.6%,6.6%,1.9%,5.1%,0.5%
Germinal.,30.7%,2.9%,6.9%,3.8%,3.2%,4.8%,6.0%,7.2%,7.9%,2.4%,7.4%,2.1%,3.1%,4.0%,3.8%,2.6%,1.1%,0.2%
L argent.,8.5%,2.7%,8.6%,6.0%,7.8%,8.7%,1.2%,5.3%,1.4%,4.2%,4.5%,5.1%,8.0%,20.2%,2.8%,1.5%,2.4%,0.7%


In [46]:
topics_par_roman = (
    best_topic_model
    .topics_per_class(
        documents,
        classes=romans,
        global_tuning=True
    )
)

display(topics_par_roman)

31it [00:00, 65.15it/s]


,Topic,Words,Frequency,Class
0,0,"abside, boutant, chasuble, arc, contrefort",97,Le reve.
1,1,"brocart, princ, tourette, tatignon, marieron",48,Le reve.
2,2,"aliment, macule, mortification, coudée, deliqu...",5,Le reve.
3,3,"brodeur, armoirie, orme, prie, entour",94,Le reve.
4,4,"mitre, coquemar, décontenancé, captif, auvent",6,Le reve.
...,...,...,...,...
543,11,"lycée, empressement, pétulant, orphelin, épigr...",22,1866 Le voeu d une morte.
544,13,"renommée, employer, rédaction, ressort, retent...",2,1866 Le voeu d une morte.
545,14,"méprit, monomanie, isolant, mathématicien, pro...",10,1866 Le voeu d une morte.
546,16,"gaucherie, laborieux, statuette, grisette, sou...",19,1866 Le voeu d une morte.


## 10. Export des résultats

In [ ]:
chemin_sortie = Path("..") /"data" /"4_resultats" /"documents_avec_topics.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)
df_resultats.to_csv(chemin_sortie, index=False, encoding="utf-8")


chemin_sortie = Path("..") /"data" /"4_resultats" /"topics_par_roman_comptages.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)
table_topics_romans_counts.to_csv(chemin_sortie, index=True, encoding="utf-8")

chemin_sortie = Path("..") /"data" /"4_resultats" /"topics_par_roman_proportions.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)
table_topics_romans_proportions.to_csv(chemin_sortie, index=True, encoding="utf-8")

chemin_sortie = Path("..") /"data" /"4_resultats" /"bertopic_topics_per_class.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)
topics_par_roman.to_csv(chemin_sortie, index=False, encoding="utf-8")



